<a href="https://colab.research.google.com/github/DeveloperPavanDesai/ai-image-generation/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install diffusers transformers accelerate torch fastapi uvicorn nest-asyncio pyngrok pillow

In [4]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")

pipe.enable_attention_slicing()
pipe.enable_model_cpu_offload()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
import uuid
import os

app = FastAPI()

os.makedirs("outputs", exist_ok=True)

class AvatarRequest(BaseModel):
    prompt: str
    steps: int = 25
    guidance: float = 7.5

@app.post("/generate-avatar")
def generate_avatar(req: AvatarRequest):
    image = pipe(
        prompt=req.prompt,
        num_inference_steps=req.steps,
        guidance_scale=req.guidance
    ).images[0]

    filename = f"{uuid.uuid4()}.png"
    filepath = f"outputs/{filename}"
    image.save(filepath)

    return {"image_path": filepath}

In [9]:
!ngrok config add-authtoken 1c40IjyAvAALuKqxmjIY0nkBuSE_2wByM7qGHqXZDwmMTWMyn

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [10]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import threading

# Function to run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start server in background thread
thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

# Start ngrok AFTER server starts
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

INFO:     Started server process [7617]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://e581-35-187-229-148.ngrok-free.app" -> "http://localhost:8000"


In [11]:
!pip install gradio

In [12]:
import gradio as gr
gr.Interface.from_pipeline(pipe).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2be47a41724587302e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
